In [ ]:
import copy
import importlib
import re

import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr
from scipy.spatial import cKDTree
import os
from typing import Union, Tuple, Optional, List, Any, Literal, Dict


workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2/model"
os.chdir(workdir)
os.getcwd()

import svensemble.vcfparse as vcfp 
import svensemble.bndbpconv as bndc
import svensemble.utils as sv_utils
import svensemble.caller_specific as caller_specific
import svensemble.benchmark as benchmark

importlib.reload(vcfp)
importlib.reload(bndc)
importlib.reload(sv_utils)
importlib.reload(caller_specific)
importlib.reload(benchmark)

In [2]:
pd.set_option('display.max_columns', None)

pu_preds_path = 'saved/model_output/y_duplessis_enoto_bps.csv'

In [ ]:
# Check if running in SLURM job by looking for SLURM environment variables
is_slurm = 'SLURM_JOB_ID' in os.environ
print(f"Running in SLURM job: {is_slurm}")


In [ ]:
# I use reference derived by algorithm described by Kosugi
# With dgv version for year 2020 instead of 2016

# val_path = '../reference/saved/NA12878.reference.tsv'
# val_df = pd.read_csv(val_path, sep='\t', index_col='Unnamed: 0')
# val_df = val_df[val_df['ALT'] == 'deletion']


val_df = benchmark.load_truth_variants('data/NA12878_DGV-2016_LR-assembly.hg38.bed')
val_vcf = benchmark.truth_to_vcf_df(val_df, ref_match_margin=50)

# ref_df_pr = ref_df_pr[(ref_df_pr['length'] > 1000) & (ref_df_pr['length'] < 100000)]
# ref_df = ref_df[ref_df['ALT'] == 'DEL']
# ref_df_pr = ref_df_pr[ref_df_pr['length'] > 50]


val_vcf_bps = benchmark.truth_event_to_bnd(val_vcf)
val_vcf_bps_dels = val_vcf_bps[val_vcf_bps['INFO_SVTYPE'] == 'DEL']
val_vcf_bps_dels_long = val_vcf_bps_dels[val_vcf_bps_dels['INFO_SVLEN'] > 50]

In [ ]:
gridss_manta_bps = pd.read_csv('saved/gridss_manta_matched.csv')
gridss_manta_bps.columns = [
    col.replace('_combined', '') for col in gridss_manta_bps
]
gridss_manta_bps

In [ ]:
# Load and parse GRIDSS VCF output
gridss_callset_path = 'data/test_gridss_output.NA12878.vcf'
if 'gridss_callset' not in locals() or 'gridss_meta' not in locals():
    gridss_callset, gridss_meta = vcfp.parse_vcf(
        gridss_callset_path,
        line_limit = None, 
        output_meta=True
    )

gridss_callset_bps = caller_specific.callset_to_bp(gridss_callset, gridss_meta, source='gridss')
# Annotate deletions in the breakpoint pairs
# gridss_callset_bps = pd.concat([gridss_callset_bps, bndc.annotate_deletions(gridss_callset_bps)], axis=1)

In [ ]:
# del manta_callset, manta_meta

if 'manta_callset' not in locals() or 'manta_meta' not in locals():
    manta_callset, manta_meta = vcfp.parse_vcf(
        'data/candidateSV.vcf.gz',
        line_limit = None, skip_format=True, 
        output_meta=True
    )


single_val_fields_for_corr =['FILTER_PASS', 'INFO_IMPRECISE', 'INFO_SVLEN',
                'INFO_MATEID', 'INFO_HOMLEN',
                'INFO_SVINSLEN']
manta_meta = sv_utils.combine_meta_dict(manta_meta)
for field in single_val_fields_for_corr:
    # print(manta_meta_sample[field])
    manta_meta[field] = list(manta_meta[field])
    # print(manta_meta_sample[field])
    manta_meta[field][0] = 1
    # print(manta_meta_sample[field])
    manta_meta[field] = tuple(manta_meta[field])
    # print(manta_meta_sample[field])

manta_meta = sv_utils.split_meta_dict(manta_meta)


In [8]:
def manta_preprocess(
    manta_callset: pd.DataFrame,
    manta_meta: pd.DataFrame,
    evidence_ci_sizes: Union[Dict[str, int], Literal['default']] = 'default',
    viable_svtypes: Union[str, List[str]] = 'simple',
    filter_pass_only: bool = True,
    inplace: bool = False
) -> Optional[Tuple[pd.DataFrame, pd.DataFrame]]:
    """Preprocess gnomAD structural variant callset.
    
    Processes a gnomAD SV callset by:
    1. Adding confidence interval sizes based on evidence type
    2. Filtering to specified SV types
    3. Optionally filtering to PASS variants only
    4. Processing evidence and metadata columns
    5. Setting EVENT field to match variant ID
    
    Args:
        gnomad_callset: DataFrame containing gnomAD SV calls
        gnomad_meta: DataFrame containing metadata for gnomAD calls
        evidence_ci_sizes: Dict mapping evidence types to CI sizes in bp, or 'default'
            Default maps SR->10bp, PE->100bp, RD->500bp
        viable_svtypes: Either 'simple' for basic SV types (DEL,DUP,INV,INS),
            or list of specific SV types to include
        filter_pass_only: If True, keep only variants with FILTER=PASS
        inplace: If True, modify input DataFrames in place
        
    Returns:
        If inplace=False: Tuple of (processed_callset, processed_meta)
        If inplace=True: None (modifies inputs in place)
        
    Raises:
        NotImplementedError: If unsupported SV types requested
    """
    # Define supported SV types
    svtypes_supported = ['DEL', 'DUP', 'INV', 'INS']

    if not inplace:
        manta_callset_local = manta_callset.copy()
        manta_meta_local = copy.deepcopy(manta_meta)
    else:
        manta_callset_local = manta_callset
        manta_meta_local = manta_meta

    manta_callset_local.loc[:, 'INFO_CHR2'] = manta_callset_local['CHROM']
    
    # Validate and set viable SV types
    if viable_svtypes == 'simple':
        viable_svtypes = svtypes_supported
    elif all(svtype in svtypes_supported for svtype in viable_svtypes):
        viable_svtypes = viable_svtypes
    else:
        raise NotImplementedError(f"Supported viable SV types: {svtypes_supported}")
    

    # Filter to desired SV types
    manta_callset_local = manta_callset_local[
        manta_callset_local['INFO_SVTYPE'].isin(viable_svtypes)
    ]
    
    # Apply FILTER=PASS filter if requested
    if filter_pass_only:
        manta_callset_local = manta_callset_local[
            manta_callset_local['FILTER_PASS'].astype(bool)
        ]
    
    
    if 'INFO_CIPOS' in manta_callset_local.columns.tolist():    
        manta_callset_cipos = (manta_callset_local['INFO_CIPOS']
                               .str.split(',', expand=True)
                               .fillna('0')
                               .astype(int))
        manta_callset_ciposl = manta_callset_cipos.loc[:, 0]
        manta_callset_ciposr = manta_callset_cipos.loc[:, 1]
    else:
        manta_callset_ciposl = manta_callset_local['INFO_CIPOS_l']
        manta_callset_ciposr = manta_callset_local['INFO_CIPOS_r']
    if 'INFO_CIEND' in manta_callset_local.columns.tolist():
        manta_callset_ciend = (manta_callset_local['INFO_CIEND']
                               .str.split(',', expand=True)
                               .fillna('0')
                               .astype(int))
        manta_callset_ciendl = manta_callset_ciend.loc[:, 0]
        manta_callset_ciendr = manta_callset_ciend.loc[:, 1]
    else:
        manta_callset_ciendl = manta_callset_local['INFO_CIEND_l']
        manta_callset_ciendr = manta_callset_local['INFO_CIEND_r']

    # Set confidence intervals based on evidence type
    match_margins = np.max([np.abs(manta_callset_ciposl), np.abs(manta_callset_ciposr), np.abs(manta_callset_ciendl), np.abs(manta_callset_ciendr)], axis=0)
    print(match_margins.shape)
    print(manta_callset_local.shape)
    manta_callset_local.loc[:, 'INFO_CIPOS_l'] = -match_margins
    manta_callset_local.loc[:, 'INFO_CIPOS_r'] = match_margins

    manta_retain = ['FILTER_PASS', 'INFO_IMPRECISE', 'INFO_SVLEN',
                'INFO_MATEID', 'INFO_HOMLEN',
                'INFO_SVINSLEN']
    
    sv_utils.delete_mult_val_cols(
        manta_callset_local,
        manta_meta_local,
        exceptions=manta_retain,
        vocal=True
    )

    sv_utils.convert_col_types(manta_callset_local, manta_meta_local)

    # Set EVENT field to match variant ID
    manta_callset_local.loc[:, 'INFO_EVENT'] = manta_callset_local['ID']

    if not inplace:
        return manta_callset_local, manta_meta_local
    else:
        manta_callset = manta_callset_local
        manta_meta = manta_meta_local
        return None

In [ ]:
manta_callset

In [ ]:
manta_callset, manta_meta = manta_preprocess(manta_callset, manta_meta, filter_pass_only=False)
manta_callset_bps = bndc.bnd_to_bp(bndc.event_to_bnd(manta_callset))
# manta_callset_bps.loc[:, 'INFO_SVLEN'] = np.abs(manta_callset_bps['INFO_SVLEN'])
if 'CHROM' in manta_callset_bps.columns.tolist():
    manta_callset_bps.loc[:, 'CHROM_bnd1'] = manta_callset_bps['CHROM']
    manta_callset_bps.loc[:, 'CHROM_bnd2'] = manta_callset_bps['CHROM']
    manta_callset_bps = manta_callset_bps.drop(columns=['CHROM'])

if 'bp_type' not in manta_callset_bps.columns.tolist():
    manta_callset_bps = sv_utils.add_bnd_encoding(manta_callset_bps, inplace=False)

if 'INFO_CIPOS_l' in manta_callset_bps.columns.tolist():
    manta_callset_bps.loc[:, 'INFO_CIPOS_l_bnd1'] = manta_callset_bps['INFO_CIPOS_l']
    manta_callset_bps.loc[:, 'INFO_CIPOS_l_bnd2'] = manta_callset_bps['INFO_CIPOS_l']
if 'INFO_CIPOS_r' in manta_callset_bps.columns.tolist():
    manta_callset_bps.loc[:, 'INFO_CIPOS_r_bnd1'] = manta_callset_bps['INFO_CIPOS_r']
    manta_callset_bps.loc[:, 'INFO_CIPOS_r_bnd2'] = manta_callset_bps['INFO_CIPOS_r']

In [ ]:
gridss_callset_bps = caller_specific.callset_to_bp(gridss_callset, gridss_meta, source='gridss')

In [12]:
def preprocess_bp_df_for_kdtree(bp_df):
    return bndc.sort_bnd_df(bp_df, chrom_keys=sv_utils.generate_chrom_key_dict(pd.concat([bp_df['CHROM_bnd1'], bp_df['CHROM_bnd2']])))

def get_chrom_bp_type(bp_df):
    """
    Utility function to get a unique identifier for chromosome position of a breakpoint + its type
    Requires a breakpoint dataframe
    """
    return bp_df['CHROM_bnd1'] + '_' + bp_df['CHROM_bnd2'] + '_' + bp_df['bp_type']

def get_kdtforest(bp_df, unique_chrom_bp_types):
    return {
        chrom_bp_type: cKDTree(bp_df[bp_df['chrom_bp_type'] == chrom_bp_type][['POS_bnd1', 'POS_bnd2']].to_numpy())
        for chrom_bp_type in unique_chrom_bp_types
    }

def kdtforest_query(kdtforest, bp_df_query, unique_chrom_bp_types_ref):
    return {
        chrom_bp_type: kdtforest[chrom_bp_type].query(bp_df_query[bp_df_query['chrom_bp_type'] == chrom_bp_type][['POS_bnd1', 'POS_bnd2']].to_numpy())
        for chrom_bp_type in unique_chrom_bp_types_ref
    }

def match_nearest_query(
    bp_df_query,
    bp_df_ref,
    ref_closest,
    unique_chrom_bp_types_query,
    chrom_bp_types_ref,
    caller_name_l='defaultcaller1',
    caller_name_r='defaultcaller2',
    reference_match=True
):
    query_matches = pd.DataFrame()
    ref_matches = pd.DataFrame()

    unmatched_chrom_bp_types = []
    for chrom_bp_type in unique_chrom_bp_types_query:
        # Filter reference breakpoints by chromosomes and breakpoint type
        filtered_bp_df_query = bp_df_query[
            bp_df_query['chrom_bp_type'] == chrom_bp_type
        ]
        filtered_bp_df_ref = bp_df_ref[
            bp_df_ref['chrom_bp_type'] == chrom_bp_type
        ]
        
        if filtered_bp_df_ref.shape[0] == 0:
            unmatched_chrom_bp_types.append(chrom_bp_type)
            continue
            
        # Reset index to ensure proper indexing when selecting matches
        filtered_bp_df_ref = filtered_bp_df_ref.reset_index(drop=True)
        
        # Get indices of closest reference breakpoints
        matched_indices = ref_closest[chrom_bp_type][1]
        
        # Select matching reference breakpoints and add to results
        matched_refs = filtered_bp_df_ref.iloc[matched_indices]
        ref_matches = pd.concat([ref_matches, matched_refs])
        query_matches = pd.concat([query_matches, filtered_bp_df_query])

    ref_matches = ref_matches.reset_index(drop=True).add_suffix(f'_{caller_name_r}')
    query_matches = query_matches.reset_index(drop=True).add_suffix(f'_{caller_name_l}')

    unmatched_calls = bp_df_query[bp_df_query['chrom_bp_type'].isin(unmatched_chrom_bp_types)].reset_index(drop=True).add_suffix(f'_{caller_name_l}')

    if not reference_match:
        matched_ref_ids = ref_matches[f'INFO_EVENT_{caller_name_r}']
        unmatched_ref = ref_matches[~ref_matches[f'INFO_EVENT_{caller_name_r}'].isin(matched_ref_ids)].reset_index(drop=True)
    else:
        unmatched_ref = pd.DataFrame()

    return pd.concat([pd.concat([query_matches, ref_matches], axis=1), unmatched_calls, unmatched_ref], axis=0).reset_index(drop=True)


def match_nearest_breakpoints(
    bp_df_query,
    bp_df_ref,
    k=1, 
    caller_name_l='defaultcaller1',
    caller_name_r='defaultcaller2',
    reference_match=True
):
    bp_df_query_preprocessed = preprocess_bp_df_for_kdtree(bp_df_query)
    bp_df_ref_preprocessed = preprocess_bp_df_for_kdtree(bp_df_ref)

    chrom_bp_types_ref = get_chrom_bp_type(bp_df_ref_preprocessed)
    chrom_bp_types_query = get_chrom_bp_type(bp_df_query_preprocessed)

    bp_df_query_preprocessed.loc[:, 'chrom_bp_type'] = chrom_bp_types_query
    bp_df_ref_preprocessed.loc[:, 'chrom_bp_type'] = chrom_bp_types_ref

    unique_chrom_bp_types_ref = chrom_bp_types_ref.unique()
    unique_chrom_bp_types_query = chrom_bp_types_query.unique()

    kdtforest = get_kdtforest(bp_df_ref_preprocessed, unique_chrom_bp_types_ref)

    ref_nearest_to_query = kdtforest_query(kdtforest, bp_df_query_preprocessed, unique_chrom_bp_types_ref)

    return match_nearest_query(bp_df_query_preprocessed, bp_df_ref_preprocessed, ref_nearest_to_query, unique_chrom_bp_types_query, unique_chrom_bp_types_ref, caller_name_l, caller_name_r, reference_match)

def get_bp_window_matches(matched_bp_df, caller_name_l='defaultcaller1', caller_name_r='defaultcaller2'):
    bnd1_left_boundary = matched_bp_df[f'POS_bnd1_{caller_name_r}'] + matched_bp_df[f'INFO_CIPOS_l_bnd1_{caller_name_r}'] + matched_bp_df[f'INFO_CIPOS_l_bnd1_{caller_name_l}']
    bnd1_right_boundary = matched_bp_df[f'POS_bnd1_{caller_name_r}'] + matched_bp_df[f'INFO_CIPOS_r_bnd1_{caller_name_r}'] + matched_bp_df[f'INFO_CIPOS_r_bnd1_{caller_name_l}']
    bnd2_left_boundary = matched_bp_df[f'POS_bnd2_{caller_name_r}'] + matched_bp_df[f'INFO_CIPOS_l_bnd2_{caller_name_r}'] + matched_bp_df[f'INFO_CIPOS_l_bnd2_{caller_name_l}']
    bnd2_right_boundary = matched_bp_df[f'POS_bnd2_{caller_name_r}'] + matched_bp_df[f'INFO_CIPOS_r_bnd2_{caller_name_r}'] + matched_bp_df[f'INFO_CIPOS_r_bnd2_{caller_name_l}']

    ref_match_bnd1_mask = (matched_bp_df[f'POS_bnd1_{caller_name_l}'] >= bnd1_left_boundary) & (matched_bp_df[f'POS_bnd1_{caller_name_l}'] <= bnd1_right_boundary)
    ref_match_bnd2_mask = (matched_bp_df[f'POS_bnd2_{caller_name_l}'] >= bnd2_left_boundary) & (matched_bp_df[f'POS_bnd2_{caller_name_l}'] <= bnd2_right_boundary)

    return ref_match_bnd1_mask & ref_match_bnd2_mask

In [ ]:
gridss_manta_bps

In [14]:
gridss_callset_bps.loc[:, 'bndtype'] = gridss_callset_bps['bnd_pos_encoding_bnd1'].astype(str) + gridss_callset_bps['bnd_pos_encoding_bnd2'].astype(str)
val_vcf_bps.loc[:, 'bndtype'] = val_vcf_bps['bnd_pos_encoding_bnd1'].astype(str) + val_vcf_bps['bnd_pos_encoding_bnd2'].astype(str)
manta_callset_bps.loc[:, 'bndtype'] = manta_callset_bps['bnd_pos_encoding_bnd1'].astype(str) + manta_callset_bps['bnd_pos_encoding_bnd2'].astype(str)
gridss_manta_bps.loc[:, 'bndtype'] = gridss_manta_bps['bp_type'].astype(str)

In [15]:
gridss_manta_bps.loc[:, 'ID_bnd1'] = gridss_manta_bps['ID_bnd1'].astype(str) + '_' + gridss_manta_bps['ID_bnd1_manta'].astype(str)
gridss_manta_bps.loc[:, 'ID_bnd2'] = gridss_manta_bps['ID_bnd2'].astype(str) + '_' + gridss_manta_bps['ID_bnd2_manta'].astype(str)
# gridss_manta_bps.loc[:, 'INFO_EVENT'] = gridss_manta_bps['INFO_EVENT'].astype(str) + '_' + gridss_manta_bps['INFO_EVENT_manta'].astype(str)

In [ ]:
gridss_manta_bps

In [17]:
# matching_bnd1, matching_bnd2 = bndc.join_bps(gridss_callset_bps, ref_vcf_bps, report_distances=True)
# matching_ids = matching_bnd1.ID[matching_bnd1.ID.isin(matching_bnd2.ID)]
# matching_bnd1_id_dist = matching_bnd1[matching_bnd1.ID.isin(matching_ids)][['ID', 'dist_to_joined']]
# matching_bnd2_id_dist = matching_bnd2[matching_bnd2.ID.isin(matching_ids)][['ID', 'dist_to_joined']]
# matching_bnd_id_dist = matching_bnd1_id_dist.merge(matching_bnd2_id_dist, on='ID', how='outer')
# matching_bnd_id_dist


In [ ]:
matching_ids_gridss = bndc.find_matching_breakpoint_ids(gridss_callset_bps, val_vcf_bps, overlap_cutoff=0.5)
matching_ids_gridss


In [19]:
# gridss_manta_bps.loc[gridss_manta_bps['INFO_CIPOS_l_bnd1'].isna(), 'INFO_CIPOS_l_bnd1'] = -20
# gridss_manta_bps.loc[gridss_manta_bps['INFO_CIPOS_r_bnd1'].isna(), 'INFO_CIPOS_r_bnd1'] = 20
# gridss_manta_bps.loc[gridss_manta_bps['INFO_CIPOS_l_bnd2'].isna(), 'INFO_CIPOS_l_bnd2'] = -20
# gridss_manta_bps.loc[gridss_manta_bps['INFO_CIPOS_r_bnd2'].isna(), 'INFO_CIPOS_r_bnd2'] = 20


In [ ]:
gridss_manta_bps

In [ ]:
matching_ids_gridss_manta = bndc.find_matching_breakpoint_ids(gridss_manta_bps, val_vcf_bps, overlap_cutoff=0.5)
matching_ids_gridss_manta

In [ ]:
matching_ids_manta = bndc.find_matching_breakpoint_ids(manta_callset_bps, val_vcf_bps, overlap_cutoff=0.5)
matching_ids_manta

In [ ]:
gridss_manta_bps['FILTER_PASS_bnd1']

In [ ]:
pu_preds_path_gridss = 'saved/model_output/y_duplessis_enoto_bps.csv'
pu_preds_path_manta = 'saved/model_output/y_duplessis_enoto_manta_bps.csv'
pu_preds_path_gridss_manta = 'saved/model_output/y_duplessis_enoto_gridss_manta_bps.csv'
preds_vs_match_gridss = pd.DataFrame(
    {
        'ID': gridss_callset_bps['INFO_EVENT'],
        'is_in_truth': gridss_callset_bps['INFO_EVENT'].isin(matching_ids_gridss),
        'QUAL': np.mean(gridss_callset_bps[['QUAL_bnd1', 'QUAL_bnd2']], axis=1),
        'FILTER_PASS': gridss_callset_bps['FILTER_PASS_bnd1'] & gridss_callset_bps['FILTER_PASS_bnd2']
    }
)
preds_vs_match_manta = pd.DataFrame(
    {
        'ID': manta_callset_bps['INFO_EVENT'],
        'is_in_truth': manta_callset_bps['INFO_EVENT'].isin(matching_ids_manta),
        'PAIR_COUNT': np.mean(manta_callset_bps[['INFO_PAIR_COUNT']], axis=1),
        'FILTER_PASS': manta_callset_bps['FILTER_PASS']
    }
)
preds_vs_match_gridss_manta = pd.DataFrame(
    {
        'ID': gridss_manta_bps['INFO_EVENT'],
        'is_in_truth': gridss_manta_bps['INFO_EVENT'].isin(matching_ids_gridss_manta),
        'QUAL': np.mean(gridss_manta_bps[['QUAL_bnd1', 'QUAL_bnd2']], axis=1),
        'PAIR_COUNT': np.mean(gridss_manta_bps[['INFO_PAIR_COUNT_manta']], axis=1),
        'FILTER_PASS': gridss_manta_bps['FILTER_PASS_bnd1'] & gridss_manta_bps['FILTER_PASS_bnd2'] & gridss_manta_bps['FILTER_PASS_manta']
    }
)
pu_preds_gridss = pd.read_csv(pu_preds_path_gridss)
pu_preds_manta = pd.read_csv(pu_preds_path_manta)
pu_preds_gridss_manta = pd.read_csv(pu_preds_path_gridss_manta)

preds_vs_match_gridss = preds_vs_match_gridss.merge(pu_preds_gridss, on='ID', how='left')
preds_vs_match_manta = preds_vs_match_manta.merge(pu_preds_manta, on='ID', how='left')
preds_vs_match_gridss_manta = preds_vs_match_gridss_manta.merge(pu_preds_gridss_manta, on='ID', how='left')
# preds_vs_match = preds_vs_match.merge(matching_bnd_id_dist, on='ID', how='left')
preds_vs_match_gridss


In [ ]:
pu_preds_gridss_manta

In [ ]:
preds_vs_match_gridss_manta

In [ ]:
preds_vs_match_test = pd.concat([preds_vs_match_gridss_manta, gridss_manta_bps['INFO_CIPOS_l_bnd1'], gridss_manta_bps['INFO_CIPOS_l_bnd2'], gridss_manta_bps['INFO_CIPOS_r_bnd1'], gridss_manta_bps['INFO_CIPOS_r_bnd2']], axis=1)
preds_vs_match_test.plot.box(by='is_in_truth', column=['y_duplessis', 'y_enoto'], figsize=(10, 5))
# preds_vs_match_test.plot.box(by='is_in_truth', column=['gridss_QUAL'], figsize=(10, 5), ylim=(0, 5000))


In [28]:
from sklearn.metrics import precision_recall_curve
from matplotlib import pyplot as plt
from sklearn.metrics import roc_curve
def plot_roc_curves(truth_series, pred_series_dict, ax=None, use_counts=True):
    """Plot ROC curves for multiple prediction arrays against a truth array.
    
    Args:
        truth_series: Series/array of ground truth binary labels
        pred_series_dict: Dict mapping prediction names to their prediction arrays
        ax: Optional matplotlib axis to plot on
        use_counts: If True, plot TP/FP counts instead of rates
    
    Returns:
        matplotlib axis with plotted ROC curves
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(4,4))
        
    for pred_name, pred_series in pred_series_dict.items():
        fpr, tpr, thresholds = roc_curve(truth_series, pred_series, pos_label=True)
        
        if use_counts:
            # Convert rates to counts
            tp = tpr * sum(truth_series)
            fp = fpr * sum(truth_series)
            ax.plot(fp, tp, label=pred_name)
        else:
            ax.plot(fpr, tpr, label=pred_name)
            
    ax.legend(loc=4)
    
    xlabel = 'False Positives' if use_counts else 'False Positive Rate'
    ylabel = 'True Positives' if use_counts else 'True Positive Rate'
    
    plt.xlabel(xlabel, size='large', weight=700)
    plt.ylabel(ylabel, size='large', weight=700)
    
    return ax


In [ ]:
pred_dict = {
    'GRIDSS': preds_vs_match_gridss.QUAL,
    'GRIDSS+SVEnsemble2\n(Du Plessis)': preds_vs_match_gridss.y_duplessis,
    'GRIDSS+SVEnsemble2\n(Elkan and Noto)': preds_vs_match_gridss.y_enoto
}

plot_roc_curves(preds_vs_match_gridss.is_in_truth, pred_dict)

In [ ]:
pred_dict = {
    'Manta': preds_vs_match_manta.PAIR_COUNT,
    'Manta+SVEnsemble2\n(Du Plessis)': preds_vs_match_manta.y_duplessis,
    'Manta+SVEnsemble2\n(Elkan and Noto)': preds_vs_match_manta.y_enoto
}

plot_roc_curves(preds_vs_match_manta.is_in_truth, pred_dict)

In [ ]:
pred_dict = {
    'GRIDSS only': preds_vs_match_gridss_manta.QUAL.fillna(0),
    'Manta only': preds_vs_match_gridss_manta.PAIR_COUNT.fillna(0),
    'GRIDSS+Manta+SVEnsemble2\n(Unbiased)': preds_vs_match_gridss_manta.y_duplessis,
    'GRIDSS+Manta+SVEnsemble2\n(Weighting)': preds_vs_match_gridss_manta.y_enoto
}

plot_roc_curves(preds_vs_match_gridss_manta.is_in_truth, pred_dict)

In [ ]:
prec_gridss, rec_gridss, thresholds_gridss = precision_recall_curve(preds_vs_match_gridss.is_in_truth, preds_vs_match_gridss.QUAL, pos_label=True, drop_intermediate=True)
prec_gridss_duplessis, rec_gridss_duplessis, thresholds_gridss_duplessis = precision_recall_curve(preds_vs_match_gridss.is_in_truth, preds_vs_match_gridss.y_duplessis, pos_label=True, drop_intermediate=True)
prec_gridss_enoto, rec_gridss_enoto, thresholds_gridss_enoto = precision_recall_curve(preds_vs_match_gridss.is_in_truth, preds_vs_match_gridss.y_enoto, pos_label=True, drop_intermediate=True)

fig, ax = plt.subplots(figsize=(4, 4))

ax.scatter(rec_gridss, prec_gridss, label='GRIDSS', s=[5]*len(rec_gridss))
ax.scatter(rec_gridss_duplessis, prec_gridss_duplessis, label='GRIDSS+SVEnsemble2\n(Du Plessis)', s=[5]*len(rec_gridss_duplessis))
ax.scatter(rec_gridss_enoto, prec_gridss_enoto, label='GRIDSS+SVEnsemble2\n(Elkan and Noto)', s=[5]*len(rec_gridss_enoto))
ax.legend(loc=1)

plt.xlabel('Recall', size='large', weight=700)
plt.ylabel('Precision', size='large', weight=700)

# plot_precision_recall_curve(ref_duplessis.is_in_ref, ref_duplessis.y_duplessis)
# print(roc_auc_score((ref_duplessis.is_in_ref, ref_duplessis.y_duplessis)))

In [ ]:
from sklearn.calibration import calibration_curve, CalibrationDisplay
prob_true, prob_pred = calibration_curve(preds_vs_match_gridss.is_in_truth, preds_vs_match_gridss.y_duplessis, n_bins=10)
disp_duplessis = CalibrationDisplay(prob_true, prob_pred, preds_vs_match_gridss.y_duplessis)

prob_true, prob_pred = calibration_curve(preds_vs_match_gridss.is_in_truth, preds_vs_match_gridss.y_enoto, n_bins=10)
disp_enoto = CalibrationDisplay(prob_true, prob_pred, preds_vs_match_gridss.y_enoto)

fig, ax = plt.subplots(figsize=(4, 4))
disp_duplessis.plot(ax=ax, label='GRIDSS+SVEnsemble2\n(Du Plessis)', color='orange')
disp_enoto.plot(ax=ax, label='GRIDSS+SVEnsemble2\n(Elkan and Noto)', color='green')

ax.legend(loc=2)

plt.xlabel('Mean predicted probability', size='large', weight=700)
plt.ylabel('Fraction of positives', size='large', weight=700)

plt.savefig('figs/calibcurve.duplessis_enoto.bps.png', bbox_inches='tight', dpi=300)

In [10]:
# gridss_callset_path = 'data/test_gridss_output.NA12878.annotated.vcf'
# gridss_callset_raw = vcfp.parse_vcf(gridss_callset_path, line_limit=None, skip_format=True)

In [11]:
# gridss_callset_raw = gridss_callset_raw[gridss_callset_raw['INFO_SIMPLE_TYPE'] == 'DEL']
# gridss_callset_raw

In [ ]:
# gridss_callset_bps_pr = gridss_callset_bps[(gridss_callset_bps['INFO_END'] - gridss_callset_bps['POS'] > 1000) & (gridss_callset_bps['INFO_END'] - gridss_callset_bps['POS'] < 100000) & (gridss_callset_bps.FILTER_PASS)]
gridss_callset_bps_pr = gridss_callset_bps[(gridss_callset_bps.length > 50)]
gridss_callset_bps_pr = gridss_callset_bps[(gridss_callset_bps.FILTER_PASS)]
gridss_callset_bps_pr

In [ ]:
gridss_callset_bps_pr = gridss_callset_bps_pr.rename(columns={
    'CHROM': 'Chromosome',
    'POS': 'Start',
    'INFO_END': 'End'
})
# gridss_callset_bps_pr = gridss_callset_bps_pr[(gridss_callset_bps_pr.End - gridss_callset_bps_pr.Start) > 50]
# gridss_callset_bps_pr = gridss_callset_bps_pr[gridss_callset_bps_pr['FILTER_PASS']]
# gridss_callset_bps_pr = gridss_callset_bps_pr[(gridss_callset_bps_pr.FILTER_PASS)]

gridss_callset_bps_pr

In [54]:
overlapped_df_path = 'saved/gridss.overlapped_bnds.csv'
callset_overlapped_df = pd.read_csv(overlapped_df_path, index_col='ID')

In [55]:
# We need to join the PU callsets to corresponding training datasets
# Since the callsets do not contain coordinate and ID information
# They need to be concated to training data first (which has the ID info)
# And to the original caller callset to fetch the coordinate inf (using the added IDs)

y_duplessis = pd.read_csv('saved/y_duplessis_bnds.csv', index_col='index', names=['index', 'y_duplessis'], skiprows=1)

In [56]:
# We need to join the PU callsets to corresponding training datasets
# Since the callsets do not contain coordinate and ID information
# They need to be concated to training data first (which has the ID info)
# And to the original caller callset to fetch the coordinate inf (using the added IDs)
y_enoto = pd.read_csv('saved/y_enoto_bnds.csv', index_col='index', names=['index', 'y_enoto'], skiprows=1)

In [57]:
callset_overlapped_df_full = pd.concat([callset_overlapped_df.reset_index(), y_duplessis, y_enoto], axis=1)[['ID', 'y_duplessis', 'y_enoto']]

In [ ]:
gridss_callset_copy = gridss_callset_raw.copy()
gridss_callset_copy = gridss_callset_copy.merge(callset_overlapped_df_full, how='left', on=['ID'])
gridss_callset_bps = bndc.bnd_to_event(gridss_callset_copy.iloc[:,], strict_matching=True, proba_cols=['y_duplessis', 'y_enoto'])
gridss_callset_bps.loc[:, 'y_duplessis_mean'] = np.mean(gridss_callset_bps[['y_duplessis_LEFT', 'y_duplessis']], axis=1)
gridss_callset_bps.loc[:, 'y_enoto_mean'] = np.mean(gridss_callset_bps[['y_enoto_LEFT', 'y_enoto']], axis=1)

In [ ]:
duplessis_callset_pr = gridss_callset_bps_pr[['Chromosome', 'Start', 'End', 'ID']].merge(callset_overlapped_df, on='ID', how='left')
duplessis_callset_pr

In [ ]:
duplessis_callset_pr.loc[:, 'length'] = duplessis_callset_pr.End - duplessis_callset_pr.Start
duplessis_callset_pr = duplessis_callset_pr[duplessis_callset_pr.length > 50]
# duplessis_callset_pr[duplessis_callset_pr]

# duplessis_callset_pr = duplessis_callset_pr[(duplessis_callset_pr.length >= 1000) & (duplessis_callset_pr.length <= 100000)]


In [29]:
# We need to join the PU callsets to corresponding training datasets
# Since the callsets do not contain coordinate and ID information
# They need to be concated to training data first (which has the ID info)
# And to the original caller callset to fetch the coordinate inf (using the added IDs)
overlapped_df_path = 'saved/gridss_overlaps.tsv'
callset_overlapped_df = pd.read_csv(overlapped_df_path, sep='\t', index_col='ID')
y_duplessis = pd.read_csv('saved/y_duplessis.csv', index_col='index', names=['index', 'y_duplessis'], skiprows=1)
callset_overlapped_df = pd.concat([callset_overlapped_df.reset_index(), y_duplessis], axis=1)[['ID', 'y_duplessis']]

duplessis_callset_pr = gridss_callset_bps_pr[['Chromosome', 'Start', 'End', 'ID']].merge(callset_overlapped_df, on='ID', how='left')
duplessis_callset_pr.loc[:, 'length'] = duplessis_callset_pr.End - duplessis_callset_pr.Start
duplessis_callset_pr = duplessis_callset_pr[duplessis_callset_pr.length > 50]
# duplessis_callset_pr[duplessis_callset_pr]

# duplessis_callset_pr = duplessis_callset_pr[(duplessis_callset_pr.length >= 1000) & (duplessis_callset_pr.length <= 100000)]


In [ ]:
duplessis_callset_pr

In [31]:
overlapped_df_path = 'saved/gridss_overlaps.tsv'
callset_overlapped_df = pd.read_csv(overlapped_df_path, sep='\t', index_col='ID')
y_enoto = pd.read_csv('saved/y_enoto.csv', index_col='index', names=['index', 'y_enoto'], skiprows=1)
callset_overlapped_df = pd.concat([callset_overlapped_df.reset_index(), y_enoto], axis=1)[['ID', 'y_enoto']]

enoto_callset_pr = gridss_callset_bps_pr[['Chromosome', 'Start', 'End', 'ID']].merge(callset_overlapped_df, on='ID', how='left')
enoto_callset_pr.loc[:, 'length'] = enoto_callset_pr.End - enoto_callset_pr.Start
enoto_callset_pr = enoto_callset_pr[enoto_callset_pr.length > 50]

# enoto_callset_pr = enoto_callset_pr[(enoto_callset_pr.length >= 1000) & (enoto_callset_pr.length <= 100000)]

In [32]:
# What I have: 
# - GRIDSS callset (for now DELs only)
# - 2 PU callsets (Trained to detect DELs)
#   
# - High confidence callset (DELs only)
# - gnomad data
#
# What I want:
# - full join of all except gnomad
#   with overlaps reported between high confidence and all other callsets
#
# How to do it:
# 1. Convert all to prs
# 2. Left-join each pr to high confidence
#    In the joined datasets there may be multiple overlaps
# 3. To resolve them, filter by overlap fraction (>80% overlap)
#    Variants with >80% overlap are considered the same, others are considered different.
#    Report all the joined variants as lists.
# 4. Concatenate the non-joined variants as if they do not overlap

In [59]:
def full_callset_join(ref_df, call_df):

    try:
        ref_df_pr = pr.PyRanges(ref_df)
    except Exception as e:
        if str(e) == 'Object is already a PyRange.':
            ref_df_pr = ref_df
            pass
        else:
            raise

    try:
        call_df_pr = pr.PyRanges(call_df)
    except Exception as e:
        if str(e) == 'Object is already a PyRange.':
            call_df_pr = call_df
            pass
        else:
            raise
    
    ref_call_pr = ref_df_pr.join(call_df_pr, report_overlap=True)
    
    ref_call_pr = ref_call_pr.df.reset_index(drop=True)
    ref_call_pr.loc[:, 'overlap_frac'] = ref_call_pr.Overlap * 2 / (ref_call_pr.End - ref_call_pr.Start + ref_call_pr.End_b - ref_call_pr.Start_b)
    ref_call_pr = ref_call_pr[ref_call_pr.overlap_frac > 0.8].sort_values(by='overlap_frac', ascending=False).drop_duplicates(subset=['ID'], keep='first').drop_duplicates(subset=['ID_b'], keep='first')
    
    # overlapping_ids_gridss_ref = ref_gridss_pr[ref_gridss_pr.overlap_frac > 0.8][['ID']]
    # overlapping_ids_gridss_callset = ref_gridss_pr[ref_gridss_pr.overlap_frac > 0.8][['ID_b']]
    
    overlapping_ids_ref = ref_call_pr['ID']
    overlapping_ids_call = ref_call_pr['ID_b']
    
    ref_df_pr_nonoverlapping = ref_df_pr.df[~ref_df_pr.df['ID'].isin(overlapping_ids_ref)].reset_index(drop=True)
    call_df_pr_nonoverlapping = call_df_pr.df[~call_df_pr.df['ID'].isin(overlapping_ids_call)].reset_index(drop=True).rename(columns={
        'ID': 'ID_b'
    })
    call_df_pr_nonoverlapping.loc[:, 'is_in_ref'] = False
    
    ref_call_pr = pd.concat([ref_call_pr, ref_df_pr_nonoverlapping, call_df_pr_nonoverlapping])
    
    # ref_call_pr.loc[:, 'ref_overlap_detected'] = (ref_call_pr.overlap_frac > 0.8) & (~ref_call_pr.overlap_frac.isna())
    # ref_call_pr.loc[:, 'is_in_ref'] = (~ref_call_pr.is_in_ref.isna()).astype(bool)

    return ref_call_pr

In [34]:
# ref_df=ref_df_pr
# call_df=gridss_callset_bps_pr

# try:
#     ref_df_pr = pr.PyRanges(ref_df)
# except Exception as e:
#     if str(e) == 'Object is already a PyRange.':
#         ref_df_pr = ref_df
#         pass
#     else:
#         raise

# try:
#     call_df_pr = pr.PyRanges(call_df)
# except Exception as e:
#     if str(e) == 'Object is already a PyRange.':
#         call_df_pr = call_df
#         pass
#     else:
#         raise

# ref_call_pr = ref_df_pr.join(call_df_pr, report_overlap=True)
# ref_call_pr = ref_call_pr.df.reset_index(drop=True)
# ref_call_pr.loc[:, 'overlap_frac'] = ref_call_pr.Overlap * 2 / (ref_call_pr.End - ref_call_pr.Start + ref_call_pr.End_b - ref_call_pr.Start_b)

# # ref_call_pr = ref_call_pr[ref_call_pr.overlap_frac > 0.8].sort_values(by='overlap_frac', ascending=False).drop_duplicates(subset=['ID'], keep='first').drop_duplicates(subset=['ID_b'], keep='first')
# ref_call_pr = ref_call_pr.sort_values(by='overlap_frac', ascending=False)#.drop_duplicates(subset=['ID'], keep='first').drop_duplicates(subset=['ID_b'], keep='first')
# ref_call_pr = ref_call_pr[ref_call_pr.overlap_frac > 0.8]

# # # overlapping_ids_gridss_ref = ref_gridss_pr[ref_gridss_pr.overlap_frac > 0.8][['ID']]
# # # overlapping_ids_gridss_callset = ref_gridss_pr[ref_gridss_pr.overlap_frac > 0.8][['ID_b']]

# # overlapping_ids_ref = ref_call_pr['ID']
# # overlapping_ids_call = ref_call_pr['ID_b']

# # ref_df_pr_nonoverlapping = ref_df_pr.df[~ref_df_pr.df['ID'].isin(overlapping_ids_ref)].reset_index(drop=True)
# # call_df_pr_nonoverlapping = call_df_pr.df[~call_df_pr.df['ID'].isin(overlapping_ids_call)].reset_index(drop=True).rename(columns={
# #     'ID': 'ID_b'
# # })
# # call_df_pr_nonoverlapping.loc[:, 'is_in_ref'] = False

# # ref_call_pr = pd.concat([ref_call_pr, ref_df_pr_nonoverlapping, call_df_pr_nonoverlapping])

# ref_call_pr.loc[:, 'length_b'] = ref_call_pr['End_b'] - ref_call_pr['Start_b']

# ref_call_pr[['Start', 'End', 'length', 'Start_b', 'End_b', 'length_b', 'Overlap', 'FILTER_PASS']]

# ref_call_pr.value_counts('FILTER_PASS')


In [35]:
ref_gridss = full_callset_join(ref_df=ref_df_pr, call_df=gridss_callset_bps_pr)
ref_gridss_h = full_callset_join(ref_df=ref_df_pr, call_df=gridss_callset_bps_pr[gridss_callset_bps_pr.FILTER_PASS.astype(bool)])
ref_duplessis = full_callset_join(ref_df=ref_df_pr, call_df=duplessis_callset_pr)
ref_enoto = full_callset_join(ref_df=ref_df_pr, call_df=enoto_callset_pr)

In [ ]:
ref_gridss = full_callset_join(ref_df=ref_df_pr, call_df=gridss_callset_raw)

In [ ]:
ref_gridss[ref_gridss['is_in_ref'] & (ref_gridss['overlap_frac'] > 0.8)]

In [36]:
# ref_gridss[ref_gridss['overlap_frac'] > 0.8]

In [37]:
ref_gridss = ref_gridss[~ref_gridss.QUAL.isna()]
ref_gridss_h = ref_gridss_h[~ref_gridss_h.QUAL.isna()]
ref_duplessis = ref_duplessis[~ref_duplessis.y_duplessis.isna()]
ref_enoto = ref_enoto[~ref_enoto.y_enoto.isna()]

In [ ]:
ref_gridss[ref_gridss.FILTER_PASS]

In [39]:
ref_gridss.loc[ref_gridss.QUAL.isna(), 'QUAL'] = -1
ref_gridss_h.loc[ref_gridss_h.QUAL.isna(), 'QUAL'] = -1
ref_duplessis.loc[ref_duplessis.y_duplessis.isna(), 'y_duplessis'] = -1
ref_enoto.loc[ref_enoto.y_enoto.isna(), 'y_enoto'] = -1
# ref_gridss_h = ref_gridss_h[~ref_gridss_h.QUAL.isna()]
# ref_duplessis = ref_duplessis[~ref_duplessis.y_duplessis.isna()]
# ref_enoto = ref_enoto[~ref_enoto.y_enoto.isna()]

In [ ]:
ref_gridss.loc[:, 'logQUAL'] = ref_gridss.QUAL
ref_gridss.boxplot(column='logQUAL', by='is_in_ref')

In [41]:
# ref_gridss = ref_gridss[
#     ((ref_gridss.End - ref_gridss.Start) > 50) & (((ref_gridss.End_b - ref_gridss.Start_b) > 50) | (ref_gridss.End_b.isna()))
# ]
# ref_duplessis = ref_duplessis[(((ref_duplessis.End - ref_duplessis.Start) > 50) & ((ref_duplessis.End_b - ref_duplessis.Start_b) > 50)) | ref_duplessis.End.isna()  | ref_duplessis.Start.isna() | ref_duplessis.End_b.isna() | ref_duplessis.Start_b.isna()]

In [42]:
test_df = pd.concat([ref_duplessis.is_in_ref, ref_duplessis.y_duplessis, ref_duplessis.ID], axis=1)

In [43]:
test_df.loc[:, 'ID_present'] = ~test_df.ID.isna()

In [ ]:
test_df

In [ ]:
test_df.boxplot(column='y_duplessis', by='is_in_ref')

In [ ]:
test_df.groupby('is_in_ref').describe()

In [47]:
def plot_roc_curve(true_y, y_prob):
    """
    plots the roc curve based of the probabilities
    """

    fpr, tpr, thresholds = roc_curve(true_y, y_prob, pos_label=True)
    plt.plot(fpr, tpr)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')

In [ ]:
roc_auc_score(ref_duplessis.is_in_ref, ref_duplessis.y_duplessis)

In [ ]:
print(
    sum(ref_gridss.is_in_ref),
    sum(ref_duplessis.is_in_ref),
    sum(ref_enoto.is_in_ref)
)

In [ ]:
fpr_gridss, tpr_gridss, thresholds_gridss = roc_curve(ref_gridss.is_in_ref, ref_gridss.QUAL, pos_label=True)
tpnum_gridss = tpr_gridss*sum(ref_gridss.is_in_ref)
fpnum_gridss = fpr_gridss*sum(ref_gridss.is_in_ref)
fpr_duplessis, tpr_duplessis, thresholds_duplessis = roc_curve(ref_duplessis.is_in_ref, ref_duplessis.y_duplessis, pos_label=True)
tpnum_duplessis = tpr_duplessis*sum(ref_gridss.is_in_ref)
fpnum_duplessis = fpr_duplessis*sum(ref_gridss.is_in_ref)
fpr_enoto, tpr_enoto, thresholds_enoto = roc_curve(ref_enoto.is_in_ref, ref_enoto.y_enoto, pos_label=True)
tpnum_enoto = tpr_enoto*sum(ref_gridss.is_in_ref)
fpnum_enoto = fpr_enoto*sum(ref_gridss.is_in_ref)

fig, ax = plt.subplots(figsize=(4,4))

ax.plot(fpnum_gridss, tpnum_gridss, label='GRIDSS')
ax.plot(fpnum_duplessis, tpnum_duplessis, label='GRIDSS+SVEnsemble2\n(Du Plessis)')
ax.plot(fpnum_enoto, tpnum_enoto, label='GRIDSS+SVEnsemble2\n(Elkan and Noto)')
# ax.plot(fpr_gridss, tpr_gridss, label='GRIDSS')
# ax.plot(fpr_duplessis, tpr_duplessis, label='GRIDSS+SVEnsemble2\n(Du Plessis)')
# ax.plot(fpr_enoto, tpr_enoto, label='GRIDSS+SVEnsemble2\n(Elkan and Noto)')
ax.legend(loc=4)

plt.xlabel('False Positives', size='large', weight=700)
plt.ylabel('True Positives', size='large', weight=700)

plt.savefig('figs/roc_full.png', bbox_inches='tight', dpi=300)

In [ ]:
plot_roc_curve(ref_gridss_h.is_in_ref, ref_gridss_h.QUAL)

In [89]:
def plot_precision_recall_curve(true_y, y_prob):
    """
    plots the roc curve based of the probabilities
    """

    prec, rec, thresholds = precision_recall_curve(true_y, y_prob)
    plt.plot(prec, rec)
    plt.xlabel('Recall')
    plt.ylabel('Precision')

In [ ]:
prec_gridss, rec_gridss, thresholds_gridss = precision_recall_curve(ref_gridss.is_in_ref, ref_gridss.QUAL, pos_label=True, drop_intermediate=True)
# tp_gridss = prec_gridss * (ref_gridss.shape[0])
prec_duplessis, rec_duplessis, thresholds_duplessis = precision_recall_curve(ref_duplessis.is_in_ref, ref_duplessis.y_duplessis, pos_label=True, drop_intermediate=True)
prec_enoto, rec_enoto, thresholds_enoto = precision_recall_curve(ref_enoto.is_in_ref, ref_enoto.y_enoto, pos_label=True, drop_intermediate=True)

fig, ax = plt.subplots(figsize=(4, 4))

ax.scatter(rec_gridss, prec_gridss, label='GRIDSS', s=[5]*len(rec_gridss))
ax.scatter(rec_duplessis,prec_duplessis, label='GRIDSS+SVEnsemble2\n(Du Plessis)', s=[5]*len(rec_duplessis))
ax.scatter(rec_enoto, prec_enoto, label='GRIDSS+SVEnsemble2\n(Elkan and Noto)', s=[5]*len(rec_enoto))
ax.legend(loc=1)

plt.xlabel('Recall', size='large', weight=700)
plt.ylabel('Precision', size='large', weight=700)

# plot_precision_recall_curve(ref_duplessis.is_in_ref, ref_duplessis.y_duplessis)
# print(roc_auc_score((ref_duplessis.is_in_ref, ref_duplessis.y_duplessis)))

In [ ]:
from sklearn.calibration import calibration_curve, CalibrationDisplay
prob_true, prob_pred = calibration_curve(ref_duplessis.is_in_ref, ref_duplessis.y_duplessis, n_bins=10)
disp_duplessis = CalibrationDisplay(prob_true, prob_pred, ref_duplessis.y_duplessis)

prob_true, prob_pred = calibration_curve(ref_enoto.is_in_ref, ref_enoto.y_enoto, n_bins=10)
disp_enoto = CalibrationDisplay(prob_true, prob_pred, ref_enoto.y_enoto)

fig, ax = plt.subplots(figsize=(4, 4))
disp_duplessis.plot(ax=ax, label='GRIDSS+SVEnsemble2\n(Du Plessis)', color='orange')
disp_enoto.plot(ax=ax, label='GRIDSS+SVEnsemble2\n(Elkan and Noto)', color='green')

ax.legend(loc=2)

plt.xlabel('Mean predicted probability', size='large', weight=700)
plt.ylabel('Fraction of positives', size='large', weight=700)

plt.savefig('figs/calibcurve.duplessis_enoto.png', bbox_inches='tight', dpi=300)

In [49]:
# y_score = classifier.decision_function(X_test)
# from sklearn.metrics import PrecisionRecallDisplay

# display = PrecisionRecallDisplay.from_predictions(
#     ref_duplessis.is_in_ref, ref_duplessis.y_duplessis, name="PU (DuPlessis correction)", plot_chance_level=True
# )
# _ = display.ax_.set_title("2-class Precision-Recall curve")


In [50]:
# display = PrecisionRecallDisplay.from_predictions(
#     ref_gridss.is_in_ref, ref_gridss.QUAL, name="GRIDSS", plot_chance_level=True
# )
# _ = display.ax_.set_title("2-class Precision-Recall curve")


In [ ]:
fig = plt.figure(figsize=(10, 10))
gs = GridSpec(4, 2)
colors = plt.get_cmap("Dark2")

ax_calibration_curve = fig.add_subplot(gs[:2, :2])
calibration_displays = {}
for i, (clf, name) in enumerate(clf_list):
    clf.fit(X_train, y_train)
    display = CalibrationDisplay.from_estimator(
        clf,
        X_test,
        y_test,
        n_bins=10,
        name=name,
        ax=ax_calibration_curve,
        color=colors(i),
    )
    calibration_displays[name] = display

ax_calibration_curve.grid()
ax_calibration_curve.set_title("Calibration plots (Naive Bayes)")

# Add histogram
grid_positions = [(2, 0), (2, 1), (3, 0), (3, 1)]
for i, (_, name) in enumerate(clf_list):
    row, col = grid_positions[i]
    ax = fig.add_subplot(gs[row, col])

    ax.hist(
        calibration_displays[name].y_prob,
        range=(0, 1),
        bins=10,
        label=name,
        color=colors(i),
    )
    ax.set(title=name, xlabel="Mean predicted probability", ylabel="Count")

plt.tight_layout()
plt.show()